# FFHQ: captions, tags and lighting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/qwen-symbolic-thinking/blob/main/notebooks/ffhq_captions.ipynb)

Every face of FFHQ (70,000, 1024 x 1024) gets a caption from the stock Qwen3.5-4B asked with `captions.PORTRAIT_ASK` (the person, the visible clothing, the background and the light), booru tags from the animetimm EVA-giant tagger (`captions.run`), and CinemaCLIP's lighting heads with the `shot light` values they support (`lighting.heads_run`). The rows ship to `raw/<stamp>/` of the dataset `AbstractPhil/ffhq-recaptioned`.

The faces come from [gaunernst/ffhq-1024-wds](https://huggingface.co/datasets/gaunernst/ffhq-1024-wds), the `images1024x1024` set of [FFHQ](https://github.com/NVlabs/ffhq-dataset) transcoded losslessly to WebP under the original numbers. FFHQ is CC BY-NC-SA 4.0 (NVIDIA); each photo keeps its Flickr author's license (FFHQ's metadata names it); not for the development of facial recognition. A row holds the image number, our caption, our tags and our lighting reading; no image.

Runtime: a GPU (the G4, RTX PRO 6000) and the `HF_TOKEN` secret with write access to the dataset repo and access to the gated animetimm tagger. Run C1, restart the session (Runtime > Restart session), then C2 (the faces into one zip), C3 (the smoke: 100 faces, read before anything else), C4 (every caption) and C5 (every lighting reading).

In [ ]:
# C1: install, then Runtime > Restart session
import importlib.util, subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("git+https://github.com/AbstractEyes/qwen-symbolic-thinking@v0.1.49", "transformers>=5.0", "accelerate>=1.0", "timm>=1.0")
pip("onnxruntime==1.23.2")   # CinemaCLIP's released ONNX encoder (lighting)

FAST_PATH = True   # the linear-attention kernels Qwen3.5 uses; causal-conv1d compiles for about 5 minutes; all or nothing
if FAST_PATH:
    if all(importlib.util.find_spec(m) for m in ("causal_conv1d", "fla", "triton")):
        print("fast path already present")
    else:
        try:
            pip("causal-conv1d", "--no-build-isolation")
            pip("flash-linear-attention", "triton")
            print("fast path on")
        except subprocess.CalledProcessError as e:
            print("fast path off (the run still works, slower):", e)
            subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "causal-conv1d"])

print("installed; now Runtime > Restart session, then run C2")

In [ ]:
# C2: the 70 WebDataset shards of FFHQ-1024 into one zip of WebP faces (each shard fetched, copied, deleted)
import os, time, zipfile
from qwen_symbolic_thinking.captions import webdataset_zip

DATA = "/content/ffhq"
IMAGES = os.path.join(DATA, "images.zip")
REPO = "AbstractPhil/ffhq-recaptioned"
TAG_OMIT = ("short hair", "see-through", "see-through sleeves", "see-through shirt", "see-through clothes", "hair bun",
            "single hair bun", "double bun", "parted lips", "lips", "realistic", "photorealistic", "asian")   # as for DeepFashion
os.makedirs(DATA, exist_ok=True)
t0 = time.time()
got = webdataset_zip("gaunernst/ffhq-1024-wds", IMAGES, work_dir=os.path.join(DATA, "shards"))
with zipfile.ZipFile(IMAGES) as zf:
    names = zf.namelist()
print(got, f"{len(names)} faces in {time.time() - t0:.0f} s; {os.path.getsize(IMAGES) / 1e9:.1f} GB; first {names[:2]}")

In [ ]:
# C3: the smoke: 100 faces with captions + tags, CinemaCLIP's lighting, and ShotVL-7B's open lighting answer; a sheet to read by eye
import io, json, os, textwrap, zipfile
from IPython.display import display
from PIL import Image, ImageDraw
from qwen_symbolic_thinking import lighting
from qwen_symbolic_thinking.captions import PORTRAIT_ASK, photos, run

N = 100
cap = run(IMAGES, None, out_repo=REPO, model_id="Qwen/Qwen3.5-4B", ask=PORTRAIT_ASK, batch=64, chunk=512, max_new_tokens=160,
          tagger="animetimm/eva_giant_patch14_560.dbv4-full", tag_omit=TAG_OMIT, tag_min_precision=0.4, n=N,
          tag="ffhq_captions_smoke100", repo_folder="raw")
light = lighting.heads_run(IMAGES, out_repo=REPO, repo_folder="raw", tag="ffhq_lighting_smoke100", n=N)
files = [os.path.basename(m) for m in photos(IMAGES, N)]
second = lighting.trial(IMAGES, files, out_repo=REPO, path_in_repo="raw", run_name="ffhq_shotvl_light_smoke100", asks=("light_open",))
print("captions", cap["path_in_repo"], cap["s_per_photo_by_op"], "| lighting", light["path_in_repo"], light["shot_light_counts"],
      light["s_per_photo"], "| ShotVL", second["stamp"], second["s_per_photo"])

def rows_of(path):
    return {r["file"]: r for r in map(json.loads, open(path, encoding="utf-8"))}
caps = rows_of(os.path.join("/content/captions", cap["path_in_repo"].split("/")[-1], "rows.jsonl"))
heads = rows_of(os.path.join(light["out_dir"], "rows.jsonl"))
sv = rows_of(os.path.join(second["out_dir"], "rows.jsonl"))
SHOW, W = files[:24], 256
sheet = Image.new("RGB", (6 * W, 4 * W), "white")
with zipfile.ZipFile(IMAGES) as zf:
    for i, f in enumerate(SHOW):
        im = Image.open(io.BytesIO(zf.read(f))).convert("RGB").resize((W, W))
        ImageDraw.Draw(im).text((6, 6), str(i + 1), fill=(255, 255, 0))
        sheet.paste(im, ((i % 6) * W, (i // 6) * W))
display(sheet)
for i, f in enumerate(SHOW):
    c, h = caps.get(f, {}), heads[f]["cinemaclip"]
    print(f"{i + 1}. {f} | shot light {heads[f]['shot_light']} | lum {heads[f]['luminance']} | cast {h['cast']['pick']} contrast "
          f"{h['contrast']['pick']} direction {h['direction']['pick']}\n   ShotVL: {sv[f]['shotvl']['light_open']['text']}\n   "
          + "\n   ".join(textwrap.wrap(c.get("caption", ""), 150)) + f"\n   tags: {c.get('tags_kept', '')[:200]}")

In [ ]:
# C4: caption and tag every face; the rows ship while the run goes (after a lost runtime: RESUME = the shipped folder)
from qwen_symbolic_thinking.captions import PORTRAIT_ASK, run

RESUME = None   # e.g. "raw/20261105_100000_ffhq_captions"
summary = run(IMAGES, None, out_repo=REPO, model_id="Qwen/Qwen3.5-4B", ask=PORTRAIT_ASK, batch=64, chunk=512, max_new_tokens=160,
              tagger="animetimm/eva_giant_patch14_560.dbv4-full", tag_omit=TAG_OMIT, tag_min_precision=0.4,
              tag="ffhq_captions", repo_folder="raw", resume=RESUME)
print(summary["path_in_repo"], summary["s_per_photo_by_op"], summary.get("missing"))

In [ ]:
# C5: CinemaCLIP's lighting heads and the shot light values for every face
from qwen_symbolic_thinking import lighting

summary = lighting.heads_run(IMAGES, out_repo=REPO, repo_folder="raw", tag="ffhq_lighting")
print(summary["path_in_repo"], summary["shot_light_counts"], summary["s_per_photo"])